# Tägliche Produkt-Filial-Abverkaufsreihen mit ADI und CV^2

Dieses Notebook klassifiziert tägliche Produkt-Filial-Abverkaufsreihen in **smooth**, **erratic**, **intermittent** und **lumpy**.

Die Analyse verwendet ausschließlich die regulären verarbeiteten Tagesdaten.


Diese Setup-Zelle importiert die benötigten Bibliotheken, definiert die Produkt-Filial-Ebene, setzt die Spalte für den Abverkauf und die ADI/CV2-Grenzwerte und lädt den verarbeiteten Tagesdatensatz.


In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

GROUP_COLS = ["ARTIKEL_ID", "MARKT_ID"]
DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
MIN_DEMAND_DAYS = 10
ADI_CUTOFF = 1.32
CV2_CUTOFF = 0.49

CLASS_ORDER = ["smooth", "erratic", "intermittent", "lumpy"]
CLASS_PALETTE = {
    "smooth": "#2a9d8f",
    "erratic": "#e9c46a",
    "intermittent": "#457b9d",
    "lumpy": "#d62828",
}

DATA_DIR_CANDIDATES = [
    Path("../../data/processed/transactions_dst_over_days"),
    Path("data/processed/transactions_dst_over_days"),
]


def resolve_data_dir(candidates):
    data_dir = next((path for path in candidates if path.exists()), None)
    if data_dir is None:
        searched = ", ".join(str(path) for path in candidates)
        raise FileNotFoundError(f"Keiner der Datenpfade wurde gefunden: {searched}")
    parquet_files = sorted(data_dir.glob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError(f"Keine Parquet-Dateien in {data_dir} gefunden")
    return data_dir


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
parquet_glob = str(DATA_DIR / "*.parquet")

print(f"Verwendete Tagesdaten: {DATA_DIR}")
print(f"Klassifikation auf Ebene: {', '.join(GROUP_COLS)}")


## Methode

Für jede Produkt-Filial-Abverkaufsreihe gilt:

- **ADI** ist `active_days / demand_days`, wobei Abverkaufstage eine verkaufte Menge größer null haben.
- **CV^2** ist `variance(non-zero demand) / mean(non-zero demand)^2`.
- **Smooth**: `ADI < 1.32` und `CV^2 < 0.49`.
- **Erratic**: `ADI < 1.32` und `CV^2 >= 0.49`.
- **Intermittent**: `ADI >= 1.32` und `CV^2 < 0.49`.
- **Lumpy**: `ADI >= 1.32` und `CV^2 >= 0.49`.

Ein aktiver Tag ist ein vorhandener Tagesdatensatz für die Produkt-Filial-Kombination. Tage mit verkaufter Menge null zählen als Null-Abverkaufstage.


Diese Zelle berechnet eine ADI/CV2-Zeile pro `ARTIKEL_ID` x `MARKT_ID`-Reihe und ordnet jede Reihe einer der Klassen `smooth`, `erratic`, `intermittent` oder `lumpy` zu.


In [ ]:
con = duckdb.connect()
con.execute("PRAGMA threads=4")

group_cols_sql = ", ".join(GROUP_COLS)
daily_group_cols_sql = ", ".join([*GROUP_COLS, "sale_date"])


def compute_series_metrics(data_dir):
    daily_parquet_glob = str(data_dir / "*.parquet")
    query = f"""
    WITH raw AS (
        SELECT
            {group_cols_sql},
            CAST(DATE AS DATE) AS sale_date,
            CAST(COALESCE({DEMAND_COL}, 0) AS DOUBLE) AS demand
        FROM read_parquet(?)
    ), daily AS (
        SELECT
            {daily_group_cols_sql},
            SUM(demand) AS demand
        FROM raw
        GROUP BY {daily_group_cols_sql}
    ), series AS (
        SELECT
            {group_cols_sql},
            COUNT(*) AS active_days,
            SUM(CASE WHEN demand > 0 THEN 1 ELSE 0 END) AS demand_days,
            SUM(demand) AS series_total_demand,
            AVG(CASE WHEN demand > 0 THEN demand END) AS mean_nonzero_demand,
            VAR_SAMP(CASE WHEN demand > 0 THEN demand END) AS var_nonzero_demand,
            MIN(sale_date) AS first_active_day,
            MAX(sale_date) AS last_active_day
        FROM daily
        GROUP BY {group_cols_sql}
    )
    SELECT
        *,
        active_days - demand_days AS zero_days,
        1 - demand_days / NULLIF(active_days, 0) AS zero_day_share,
        active_days / NULLIF(demand_days, 0) AS ADI,
        CASE
            WHEN demand_days > 1 AND mean_nonzero_demand > 0
                THEN var_nonzero_demand / (mean_nonzero_demand * mean_nonzero_demand)
            WHEN demand_days = 1 THEN 0.0
            ELSE NULL
        END AS CV2
    FROM series
    WHERE demand_days > 0
    """
    return con.execute(query, [daily_parquet_glob]).fetchdf()


def classify_demand(metrics):
    conditions = [
        (metrics["ADI"] < ADI_CUTOFF) & (metrics["CV2"] < CV2_CUTOFF),
        (metrics["ADI"] < ADI_CUTOFF) & (metrics["CV2"] >= CV2_CUTOFF),
        (metrics["ADI"] >= ADI_CUTOFF) & (metrics["CV2"] < CV2_CUTOFF),
        (metrics["ADI"] >= ADI_CUTOFF) & (metrics["CV2"] >= CV2_CUTOFF),
    ]
    classified = metrics.assign(
        demand_class=np.select(conditions, CLASS_ORDER, default="unclassified")
    )
    classified = classified[classified["demand_class"].isin(CLASS_ORDER)].copy()
    classified = classified[classified["demand_days"] >= MIN_DEMAND_DAYS].copy()
    classified["demand_class"] = pd.Categorical(
        classified["demand_class"], categories=CLASS_ORDER, ordered=True
    )
    return classified


series_metrics = compute_series_metrics(DATA_DIR)
series_classification = classify_demand(series_metrics)
series_classification.head()


Diese Übersicht zeigt die Anzahl der Reihen, Produkte, Filialen, aktiven Tage, Abverkaufstage, Null-Abverkaufstage sowie die Medianwerte für ADI und CV2.


In [ ]:
overall_summary = pd.DataFrame([
    {
        "series": len(series_classification),
        "products": series_classification["ARTIKEL_ID"].nunique(),
        "stores": series_classification["MARKT_ID"].nunique(),
        "active_days": series_classification["active_days"].sum(),
        "demand_days": series_classification["demand_days"].sum(),
        "zero_days": series_classification["zero_days"].sum(),
        "median_zero_day_share": series_classification["zero_day_share"].median(),
        "median_ADI": series_classification["ADI"].median(),
        "median_CV2": series_classification["CV2"].median(),
    }
])
overall_summary["zero_day_share_%"] = (
    100 * overall_summary["zero_days"] / overall_summary["active_days"]
).round(1)
overall_summary["median_zero_day_share_%"] = (100 * overall_summary["median_zero_day_share"]).round(1)
overall_summary[["median_ADI", "median_CV2"]] = overall_summary[["median_ADI", "median_CV2"]].round(2)
overall_summary.drop(columns="median_zero_day_share")


Diese Tabelle aggregiert die klassifizierten Reihen nach ADI/CV2-Klasse. Sie zeigt, wie viele Reihen in jede Klasse fallen und wie dünn oder variabel diese Klassen im Median sind.


In [ ]:
class_summary = (
    series_classification
    .groupby("demand_class", observed=True)
    .agg(
        series=("ARTIKEL_ID", "size"),
        products=("ARTIKEL_ID", "nunique"),
        active_days=("active_days", "sum"),
        demand_days=("demand_days", "sum"),
        median_ADI=("ADI", "median"),
        median_CV2=("CV2", "median"),
        median_zero_day_share=("zero_day_share", "median"),
    )
    .reindex(CLASS_ORDER)
    .reset_index()
)
class_summary[["series", "products", "active_days", "demand_days"]] = (
    class_summary[["series", "products", "active_days", "demand_days"]]
    .fillna(0)
    .astype(int)
)
total_series = class_summary["series"].sum()
class_summary["series_share"] = np.where(
    total_series > 0,
    class_summary["series"] / total_series,
    0,
)

class_summary_display = class_summary.copy()
class_summary_display["series_share_%"] = (100 * class_summary_display["series_share"]).round(1)
class_summary_display["median_zero_day_share_%"] = (100 * class_summary_display["median_zero_day_share"]).round(1)
class_summary_display[["median_ADI", "median_CV2"]] = class_summary_display[["median_ADI", "median_CV2"]].round(2)
class_summary_display = class_summary_display[
    [
        "demand_class", "series", "products", "active_days",
        "demand_days", "series_share_%", "median_zero_day_share_%", "median_ADI", "median_CV2",
    ]
]
class_summary_display


Diese kompakte Tabelle übersetzt die Medianwerte von ADI und CV2 in die praktische Bedeutung jeder ADI/CV2-Klasse.


In [ ]:
class_interpretations = {
    "smooth": "häufiger Abverkauf mit stabiler Menge",
    "erratic": "häufiger Abverkauf mit variabler Menge",
    "intermittent": "seltener Abverkauf mit stabiler Menge",
    "lumpy": "seltener Abverkauf mit variabler Menge",
}

class_interpretation_table = (
    class_summary
    .set_index("demand_class")
    .reindex(CLASS_ORDER)
    .reset_index()[["demand_class", "median_ADI", "median_CV2"]]
)
class_interpretation_table["interpretation"] = (
    class_interpretation_table["demand_class"].map(class_interpretations)
)
class_interpretation_table[["median_ADI", "median_CV2"]] = (
    class_interpretation_table[["median_ADI", "median_CV2"]].round(2)
)

class_interpretation_table


Dieser Plot visualisiert die Anteile der täglichen Produkt-Filial-Abverkaufsreihen je ADI/CV2-Klasse.


In [ ]:
plot_summary = class_summary.set_index("demand_class").reindex(CLASS_ORDER).reset_index()

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(
    plot_summary["demand_class"],
    plot_summary["series_share"],
    color=[CLASS_PALETTE[demand_class] for demand_class in plot_summary["demand_class"]],
)

ax.set_title("Anteil täglicher Produkt-Filial-Abverkaufsreihen")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")

plt.tight_layout()


Diese Tabelle gruppiert `smooth` und `erratic` als regelmäßige Abverkaufsreihen sowie `intermittent` und `lumpy` als seltene Abverkaufsreihen.


In [ ]:
regular_sparse_share_table = (
    class_summary
    .assign(
        demand_frequency=np.where(
            class_summary["demand_class"].isin(["smooth", "erratic"]),
            "regelmäßige Abverkaufsreihen (%)",
            "seltene Abverkaufsreihen (%)",
        )
    )
    .groupby("demand_frequency", observed=True)["series_share"]
    .sum()
    .mul(100)
    .round(1)
    .reindex(["regelmäßige Abverkaufsreihen (%)", "seltene Abverkaufsreihen (%)"])
    .to_frame()
    .T
)
regular_sparse_share_table.index = ["Anteil"]

regular_sparse_share_table


Diese Tabelle zeigt Quartile für ADI, CV2, aktive Tage, Abverkaufstage und Null-Abverkaufsanteil nach Klasse. Sie hilft einzuschätzen, ob Klassen durch typische Verläufe oder durch Grenzfälle geprägt sind.


In [ ]:
quantiles = (
    series_classification
    .groupby("demand_class", observed=True)[["ADI", "CV2", "active_days", "demand_days", "zero_day_share"]]
    .quantile([0.25, 0.50, 0.75])
    .round(2)
)
quantiles


## Vier repräsentative tägliche Abverkaufsreihen

Die vier Beispiele wählen je eine Produkt-Filial-Reihe pro ADI/CV2-Klasse aus. Die Auswahl folgt der gleichen Idee wie im Notebook zur wöchentlichen Heterogenität: stabile Artikelmetadaten werden bevorzugt, und zuerst wird versucht, dasselbe Produkt über verschiedene Filialen hinweg zu zeigen.


Diese Zelle wählt vier repräsentative Produkt-Filial-Reihen aus den Tagesdaten aus, jeweils eine pro ADI/CV2-Klasse. Nach Möglichkeit wird dasselbe Produkt über verschiedene Filialen verwendet, damit die Panels Filialunterschiede für ein vergleichbares Produkt zeigen.


In [ ]:
PREFERRED_MIN_ACTIVE_DAYS = 120
daily_glob = parquet_glob

daily_classification = series_classification.copy()
daily_classification["demand_class"] = daily_classification["demand_class"].astype(str)

daily_class_medians = (
    daily_classification
    .groupby("demand_class", observed=True)[["ADI", "CV2"]]
    .median()
    .rename(columns={"ADI": "class_median_ADI", "CV2": "class_median_CV2"})
)

daily_metadata = con.execute(
    """
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        MIN(ARTIKEL_BEZ) AS ARTIKEL_BEZ,
        MIN(ARTIKEL_INHALT) AS ARTIKEL_INHALT,
        MIN(VERKAUFSEINHEIT) AS VERKAUFSEINHEIT,
        MIN(GEWICHTSARTIKEL) AS GEWICHTSARTIKEL,
        COUNT(DISTINCT ARTIKEL_INHALT) AS n_artikel_inhalt,
        COUNT(DISTINCT VERKAUFSEINHEIT) AS n_verkaufseinheit,
        COUNT(DISTINCT GEWICHTSARTIKEL) AS n_gewichtsartikel
    FROM read_parquet(?)
    GROUP BY ARTIKEL_ID, MARKT_ID
    """,
    [daily_glob],
).fetchdf()

daily_examples_base = daily_classification.merge(
    daily_metadata, on=GROUP_COLS, how="left"
)

stable_metadata = (
    (daily_examples_base["n_artikel_inhalt"] == 1)
    & (daily_examples_base["n_verkaufseinheit"] == 1)
    & (daily_examples_base["n_gewichtsartikel"] == 1)
)
daily_metadata_candidates = daily_examples_base[stable_metadata].copy()


def score_daily_example_candidates(candidates):
    scored = candidates.merge(daily_class_medians, on="demand_class", how="left").copy()
    scored["median_distance"] = (
        (scored["ADI"] - scored["class_median_ADI"]).abs()
        / scored["class_median_ADI"].replace(0, np.nan)
        + (scored["CV2"] - scored["class_median_CV2"]).abs()
        / scored["class_median_CV2"].replace(0, np.nan)
    )
    return scored


def has_all_daily_classes(candidates):
    return set(candidates["demand_class"].dropna()).issuperset(CLASS_ORDER)


def select_same_daily_article(candidates):
    if candidates.empty or not has_all_daily_classes(candidates):
        return pd.DataFrame()

    scored = score_daily_example_candidates(candidates)
    best_per_article_class = (
        scored
        .sort_values(
            ["ARTIKEL_ID", "demand_class", "median_distance", "active_days"],
            ascending=[True, True, True, False],
        )
        .groupby(["ARTIKEL_ID", "demand_class"], observed=True)
        .head(1)
    )

    article_scores = (
        best_per_article_class
        .groupby("ARTIKEL_ID", observed=True)
        .agg(
            n_classes=("demand_class", "nunique"),
            total_distance=("median_distance", "sum"),
            median_active_days=("active_days", "median"),
            ARTIKEL_BEZ=("ARTIKEL_BEZ", "first"),
            ARTIKEL_INHALT=("ARTIKEL_INHALT", "first"),
        )
        .loc[lambda df: df["n_classes"].eq(len(CLASS_ORDER))]
        .sort_values(["total_distance", "median_active_days"], ascending=[True, False])
    )

    if article_scores.empty:
        return pd.DataFrame()

    selected_article_id = article_scores.index[0]
    return (
        best_per_article_class[best_per_article_class["ARTIKEL_ID"].eq(selected_article_id)]
        .set_index("demand_class")
        .loc[CLASS_ORDER]
        .reset_index()
    )


def select_daily_per_class(candidates):
    if candidates.empty or not has_all_daily_classes(candidates):
        return pd.DataFrame()

    return (
        score_daily_example_candidates(candidates)
        .sort_values(
            ["demand_class", "median_distance", "active_days"],
            ascending=[True, True, False],
        )
        .groupby("demand_class", observed=True)
        .head(1)
        .set_index("demand_class")
        .loc[CLASS_ORDER]
        .reset_index()
    )


daily_candidate_pools = [
    (
        "gleiches Produkt in vier Filialen/Klassen, stabile Metadaten, active_days >= "
        f"{PREFERRED_MIN_ACTIVE_DAYS}",
        daily_metadata_candidates[
            daily_metadata_candidates["active_days"] >= PREFERRED_MIN_ACTIVE_DAYS
        ],
        select_same_daily_article,
    ),
    (
        "gleiches Produkt in vier Filialen/Klassen, stabile Metadaten",
        daily_metadata_candidates,
        select_same_daily_article,
    ),
    (
        "eine stabile Reihe pro Klasse, active_days >= "
        f"{PREFERRED_MIN_ACTIVE_DAYS}",
        daily_metadata_candidates[
            daily_metadata_candidates["active_days"] >= PREFERRED_MIN_ACTIVE_DAYS
        ],
        select_daily_per_class,
    ),
    ("eine stabile Reihe pro Klasse", daily_metadata_candidates, select_daily_per_class),
    ("eine Reihe pro Klasse", daily_examples_base, select_daily_per_class),
]

for daily_selection_note, candidates, selector in daily_candidate_pools:
    daily_example_series = selector(candidates)
    if not daily_example_series.empty:
        break
else:
    available = (
        daily_classification.groupby("demand_class", observed=True)
        .size()
        .reindex(CLASS_ORDER)
        .fillna(0)
        .astype(int)
    )
    raise ValueError(
        "Es konnte nicht je ein tägliches Beispiel pro Klasse ausgewählt werden. "
        f"Verfügbare Reihen je Klasse: {available.to_dict()}"
    )

daily_example_series = daily_example_series.copy()
daily_example_series["store_label"] = [
    f"Filiale {chr(65 + i)}" for i in range(len(daily_example_series))
]

daily_example_display_cols = [
    "demand_class", "store_label", "ARTIKEL_ID", "MARKT_ID",
    "ARTIKEL_BEZ", "ARTIKEL_INHALT", "VERKAUFSEINHEIT",
    "GEWICHTSARTIKEL", "active_days", "demand_days",
    "ADI", "class_median_ADI", "CV2", "class_median_CV2",
    "median_distance",
]

print(f"Auswahl: {daily_selection_note}")
daily_example_series[daily_example_display_cols].round({
    "ADI": 2,
    "class_median_ADI": 2,
    "CV2": 2,
    "class_median_CV2": 2,
    "median_distance": 3,
})


## Tägliche Beispielverläufe

Jedes Panel zeigt die täglich verkaufte Menge einer ausgewählten Produkt-Filial-Reihe. Filiallabels sind anonymisiert, damit die Form der Abverkaufsreihe im Vordergrund bleibt.


Dieser Vier-Panel-Plot zeigt die ausgewählten täglichen Abverkaufsreihen. Jedes Panel enthält eine ADI/CV2-Klasse; die y-Achse verwendet die jeweilige `VERKAUFSEINHEIT`.


In [ ]:
daily_example_keys = daily_example_series[
    ["demand_class", "store_label", "ARTIKEL_ID", "MARKT_ID"]
].copy()
con.register("daily_example_keys", daily_example_keys)

daily_examples_query = f"""
SELECT
    k.demand_class,
    k.store_label,
    d.ARTIKEL_ID,
    d.MARKT_ID,
    CAST(d.DATE AS DATE) AS sale_date,
    SUM(CAST(COALESCE(d.{DEMAND_COL}, 0) AS DOUBLE)) AS demand
FROM read_parquet(?) AS d
JOIN daily_example_keys AS k
    ON d.ARTIKEL_ID = k.ARTIKEL_ID
    AND d.MARKT_ID = k.MARKT_ID
GROUP BY
    k.demand_class, k.store_label, d.ARTIKEL_ID, d.MARKT_ID, sale_date
ORDER BY k.demand_class, k.store_label, sale_date
"""
daily_example_ts = con.execute(
    daily_examples_query, [daily_glob]
).fetchdf()

daily_plot_data = daily_example_ts.merge(
    daily_example_series[[
        "demand_class", "store_label", "ARTIKEL_ID", "MARKT_ID",
        "ARTIKEL_BEZ", "ARTIKEL_INHALT", "VERKAUFSEINHEIT",
        "ADI", "CV2", "active_days", "demand_days",
    ]],
    on=["demand_class", "store_label", "ARTIKEL_ID", "MARKT_ID"],
    how="left",
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=False)
axes = axes.ravel()

for ax, demand_class in zip(axes, CLASS_ORDER):
    subset = daily_plot_data[
        daily_plot_data["demand_class"].eq(demand_class)
    ].sort_values("sale_date")
    if subset.empty:
        ax.axis("off")
        ax.set_title(f"{demand_class.capitalize()}\nKeine Daten")
        continue

    meta = subset.iloc[0]
    color = CLASS_PALETTE[demand_class]
    positive_subset = subset[subset["demand"] > 0]
    article_name = str(meta["ARTIKEL_BEZ"])
    if len(article_name) > 44:
        article_name = article_name[:41] + "..."

    ax.plot(subset["sale_date"], subset["demand"], color=color, linewidth=1.1, alpha=0.85)
    ax.scatter(
        positive_subset["sale_date"], positive_subset["demand"],
        color=color, s=12, alpha=0.75,
    )
    ax.axhline(0, color="#444444", linewidth=0.8, alpha=0.5)
    ax.set_title(
        f"{demand_class.capitalize()} · {meta['store_label']}\n"
        f"{article_name}\n"
        f"ADI={meta['ADI']:.2f}, CV2={meta['CV2']:.2f}"
    )
    ax.set_xlabel("Datum")
    ax.set_ylabel(f"Täglich verkaufte Menge, kg")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()


## Interpretation

Die tägliche Aggregation macht sichtbar, wie unterschiedlich Produkt-Filial-Abverkaufsreihen auch innerhalb derselben Datenbasis ausfallen. `Smooth` und `erratic` stehen für häufige Abverkaufstage, während `intermittent` und `lumpy` stärkere Lücken zwischen Verkäufen zeigen.

Hohe Null-Abverkaufsanteile erhöhen den ADI und verschieben Reihen in seltenere Klassen. Deshalb ist die tägliche Klassifikation besonders empfindlich gegenüber langen Abschnitten ohne Verkäufe.
